## Goal
Pack 14 Christmas tree polygons into the smallest square. Target score: 0.3696

Score formula: `max(width, height)^2 / N`

In [ ]:
!git clone https://github.com/SmartManoj/sparrow
!curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y
!bash -lc "source $HOME/.cargo/env && rustup default nightly"

# build once
!bash -lc "source $HOME/.cargo/env && cd sparrow && cargo build --release --features=simd,only_final_svg"


In [ ]:
# ============================================================
# Sparrow Multi-Restart Runner for Santa2025 (N=57/58/65/151)
# - Use high-score side_length as height constraint (scaled *1000)
# - Restart with multiple seeds + length window
# - Auto-merge improvements into submission.csv
# ============================================================

import os, json, math, random, subprocess, shutil
from pathlib import Path

# ----------------------------
# BASIC CONFIG
# ----------------------------
SCALE = 1000.0                 # same as your notebook
SPARROW_DIR = Path("sparrow")  # cloned repo folder
OUTPUT_DIR = SPARROW_DIR / "output"

# Time per run (seconds)
TIME_LIMIT_SEC = 120  # 60/120/300都可以

# How many restarts per length
RESTARTS_PER_LENGTH = 6

# Seeds to try (you can add more)
BASE_SEEDS = [42, 100, 200, 300, 400, 777, 999]

# Length window: L*, L*+0.01, L*+0.02
L_OFFSETS = [0.00, 0.01, 0.02]

# Targets from your high-score table (side_length)
TARGET = {
    57: 4.484638629210352640,
    58: 4.496813336078086656,
    65: 4.781086052083545088,
    151: 7.111531351091566592,
}

# Input/Output submission
BASE_SUBMISSION = Path("/kaggle/input/intergration-of-existing-result-current-best/submission.csv")
OUT_SUBMISSION  = Path("submission_sparrow_merge.csv")

assert SPARROW_DIR.exists(), f"❌ {SPARROW_DIR} not found, please clone sparrow first."
assert BASE_SUBMISSION.exists(), f"❌ {BASE_SUBMISSION} not found, please copy it first."

print("✅ Sparrow folder and submission.csv found.")
print(f"TIME_LIMIT_SEC={TIME_LIMIT_SEC}, RESTARTS_PER_LENGTH={RESTARTS_PER_LENGTH}")

# ----------------------------
# TREE POLYGON (same as your notebook)
# ----------------------------
TREE_VERTS = [
    (0.0, 0.8), (0.125, 0.5), (0.0625, 0.5), (0.2, 0.25), (0.1, 0.25),
    (0.35, 0.0), (0.075, 0.0), (0.075, -0.2), (-0.075, -0.2), (-0.075, 0.0),
    (-0.35, 0.0), (-0.1, 0.25), (-0.2, 0.25), (-0.0625, 0.5), (-0.125, 0.5)
]

TREE_VERTS_SCALED = [[vx * SCALE, vy * SCALE] for vx, vy in TREE_VERTS]

# ----------------------------
# Score calculation (same logic as your notebook)
# ----------------------------
def transform_point(x, y, tx, ty, deg):
    rad = math.radians(deg)
    c, s = math.cos(rad), math.sin(rad)
    rx = x * c - y * s
    ry = x * s + y * c
    return rx + tx, ry + ty

def calc_score(placements):
    """placements: [(x, y, deg), ...]"""
    min_x = min_y = float('inf')
    max_x = max_y = float('-inf')

    for tx, ty, deg in placements:
        for vx, vy in TREE_VERTS:
            px, py = transform_point(vx, vy, tx, ty, deg)
            min_x = min(min_x, px)
            max_x = max(max_x, px)
            min_y = min(min_y, py)
            max_y = max(max_y, py)

    width = max_x - min_x
    height = max_y - min_y
    side = max(width, height)
    return side ** 2 / len(placements)

# ----------------------------
# submission.csv helpers
# ----------------------------
def load_submission(csv_path):
    data = {}
    with open(csv_path, "r", encoding="utf-8") as f:
        f.readline()  # header
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split(',')
            id_ = parts[0]
            x = float(parts[1][1:])  # remove 's'
            y = float(parts[2][1:])
            deg = float(parts[3][1:])
            data[id_] = (x, y, deg)
    return data

def extract_group(data, group_str):
    placements = []
    i = 0
    while f"{group_str}_{i}" in data:
        placements.append(data[f"{group_str}_{i}"])
        i += 1
    return placements

def write_submission(data, out_path):
    def sort_key(k):
        g, idx = k.split('_')
        return (int(g), int(idx))
    keys = sorted(data.keys(), key=sort_key)
    with open(out_path, "w", encoding="utf-8") as f:
        f.write("id,x,y,deg\n")
        for k in keys:
            x, y, deg = data[k]
            f.write(f"{k},s{x:.17f},s{y:.17f},s{deg:.17f}\n")

# ----------------------------
# sparrow io helpers
# ----------------------------
def build_sparrow_input_json(n: int, height_int: int, out_json_path: Path):
    """Create sparrow input json: 1 item with demand=n, polygon scaled *1000."""
    data = {
        "name": f"n{n}_h{height_int}",
        "items": [{
            "id": 0,
            "demand": int(n),
            "shape": {
                "type": "simple_polygon",
                "data": TREE_VERTS_SCALED
            }
        }],
        "strip_height": int(height_int)
    }
    with open(out_json_path, "w", encoding="utf-8") as f:
        json.dump(data, f)
    return data["name"]

def load_sparrow_solution(final_json_path: Path):
    """Return placements [(x,y,deg)...] from sparrow output json."""
    with open(final_json_path, "r", encoding="utf-8") as f:
        data = json.load(f)
    sol = data.get("solution", data)

    placements = []
    for item in sol["layout"]["placed_items"]:
        t = item["transformation"]
        x = t["translation"][0] / SCALE
        y = t["translation"][1] / SCALE
        deg = t["rotation"]
        placements.append((x, y, deg))
    return placements

def run_sparrow_once(input_json: Path, t_sec: int, seed: int):
    """Run sparrow and return final json path if exists."""
    cmd = [
        "bash", "-lc",
        f"cd {SPARROW_DIR} && cargo run --release --features=simd,only_final_svg -- "
        f"-i ../{input_json} -t {t_sec} -s {seed}"
    ]
    # run
    subprocess.run(cmd, check=True)

    # output name is taken from input json "name"
    with open(input_json, "r", encoding="utf-8") as f:
        name = json.load(f)["name"]

    final_json = OUTPUT_DIR / f"final_{name}.json"
    final_svg  = OUTPUT_DIR / f"final_{name}.svg"
    return final_json, final_svg

# ----------------------------
# Main optimization loop
# ----------------------------
# Copy base submission to working buffer
work_data = load_submission(BASE_SUBMISSION)

def try_merge_group(n: int, sparrow_placements):
    """Merge into work_data if better."""
    group_str = f"{n:03d}"

    old_placements = extract_group(work_data, group_str)
    if not old_placements:
        print(f"⚠️ Group {group_str} not found in submission.csv")
        return False, None, None

    old_score = calc_score(old_placements)
    new_score = calc_score(sparrow_placements)

    improved = new_score < old_score
    return improved, old_score, new_score

# summary
best_record = {}  # n -> (best_score, best_info)

print("\n" + "="*70)
print("🚀 START: Sparrow Strategic Restarts")
print("="*70)

for n, L_star in TARGET.items():
    group_str = f"{n:03d}"
    old_placements = extract_group(work_data, group_str)
    base_score = calc_score(old_placements)

    print("\n" + "-"*70)
    print(f"🎯 N={n} (group {group_str})")
    print(f"Current score in submission: {base_score:.12f}")
    print(f"High-score side_length L*   : {L_star:.12f}")
    print("-"*70)

    local_best = (base_score, None, None)  # (score, final_json, (height, seed, offset))

    # length window
    for off in L_OFFSETS:
        L = L_star + off
        height_int = int(round(L * SCALE))

        # multi restarts
        for r in range(RESTARTS_PER_LENGTH):
            seed = random.choice(BASE_SEEDS) + r*17 + int(off*1000)

            input_json = Path(f"n{n:03d}_h{height_int}_seed{seed}.json")
            name = build_sparrow_input_json(n, height_int, input_json)

            try:
                final_json, final_svg = run_sparrow_once(input_json, TIME_LIMIT_SEC, seed)
            except Exception as e:
                print(f"  ❌ Sparrow failed: N={n} height={height_int} seed={seed} err={e}")
                continue

            if not final_json.exists():
                print(f"  ❌ Missing output json: {final_json}")
                continue

            placements = load_sparrow_solution(final_json)

            improved, old_score, new_score = try_merge_group(n, placements)
            print(f"  [off={off:+.2f}] h={height_int} seed={seed} -> new={new_score:.12f}  (old={old_score:.12f})")

            # if improved: merge into work_data immediately
            if improved:
                # replace group
                for i, (x, y, deg) in enumerate(placements):
                    work_data[f"{group_str}_{i}"] = (x, y, deg)

                print(f"    ✅ IMPROVED! merged into working submission. Δ={old_score-new_score:.12f}")

                # track best
                if new_score < local_best[0]:
                    local_best = (new_score, str(final_json), (height_int, seed, off))

    best_record[n] = local_best

# Write final merged submission
write_submission(work_data, OUT_SUBMISSION)

print("\n" + "="*70)
print("✅ DONE. Summary of best improvements")
print("="*70)

for n in sorted(best_record.keys()):
    best_score, best_json, info = best_record[n]
    group_str = f"{n:03d}"
    print(f"N={n:3d} group={group_str} best_score={best_score:.12f}  best_json={best_json}  info={info}")

print(f"\n📌 Final merged submission saved: {OUT_SUBMISSION}")
